# 11 · Run with a preload: adopt what is valid, compute the rest

**When:** the Right side is ready. A normal run starts, Stage 1 runs live on both sides, and
before `col-check` runs, `handoff.adopt_preloaded` commits every preloaded unit that still fits.
`col-check` then computes only what is left, because it skips units already in `_INDEX/`.

```text
row-check (live)  ->  Right reconcile + gate + manifests  ->  adopt_preloaded  ->  col-check (the rest)  ->  compare  ->  report
```

A preloaded unit is adopted only when **all** of these hold for the run's manifest window:

| Check | Fails when |
| --- | --- |
| same partition set | the Right side excluded a partition, so the window has a hole |
| same count per partition (live row-check vs preload time) | the source gained or lost rows |
| same fingerprint (sample, keys, key families, strict, vintage, left source) | settings changed |
| same `col_type_overrides` entry for the column | a type override was added or changed |

In [ ]:
import handoff
import nbkit
import pandas as pd
from nbkit import table, tree

from dtrack_left.checks.col_check import ColCheckPipeline, LeftRuntime, wait_for_manifests
from dtrack_right.checks.col_check import ColCheck

folder = nbkit.sandbox("11")
data, config, _ = nbkit.dataset(folder)
full = {"pairs": {"orders": config["pairs"]["orders"]}}
left_only = {
    "pairs": {
        "orders": {
            k: v for k, v in config["pairs"]["orders"].items() if k not in ("right", "col_map")
        }
    }
}
left_only["pairs"]["orders"]["columns"] = list(config["pairs"]["orders"]["col_map"])
pair = handoff.pair_from_config(left_only, "orders")


def run(name, preload_store=None):
    store = nbkit.VolumeStore(folder / "runs" / name)
    nbkit.left_row_check(full, store, run_id=name)
    nbkit.right_row_check(store, nbkit.target_factory(data, folder))
    decisions = handoff.adopt_preloaded(store, preload_store) if preload_store else None
    runtime = LeftRuntime(folder / f"left_work_{name}", store)
    manifests = wait_for_manifests(store, timeout=1, poll_interval=0)
    computed = ColCheckPipeline(
        store, pull_window=runtime.pull_window, compute_stats=runtime.compute_stats
    ).run(manifests)
    results = ColCheck(store, nbkit.target_factory(data, folder), poll_interval=0).run()
    verdicts = {
        (row["vintage"], row["column"]): (row["verdict"], row["hash_sum_left"]) for row in results
    }
    return store, decisions, computed, verdicts

## 1 · Preloaded without knowing the matches

All Left partitions were preloaded. Stage 1 then finds a count diff on 01-15, a Right-only day
01-31 and a Left-only day 02-10, so both month windows have holes: nothing is adoptable and
`col-check` computes all units. The preload was wasted, but the result is still correct.

In [ ]:
blind = nbkit.VolumeStore(folder / "preload_blind")
handoff.preload_units(blind, "orders", pair, folder / "preload_work", handoff.left_counts(pair))
store, decisions, computed, blind_verdicts = run("blind", blind)
table(decisions)
print("computed by col-check:", computed)

## 2 · Preloaded with the known matched partitions

Here the matched partitions are known up front (taken from the verdicts above; in practice from
an earlier run or a quick Right-side count). Every unit is adopted and `col-check` computes
nothing.

In [ ]:
matched = [
    day
    for day, row in store.get_json("row_check/orders.json")["partitions"].items()
    if row["verdict"] == "match"
]
known = nbkit.VolumeStore(folder / "preload_known")
handoff.preload_units(
    known, "orders", pair, folder / "preload_work", handoff.left_counts(pair, partitions=matched)
)
store, decisions, computed, known_verdicts = run("known", known)
table(decisions)
print("computed by col-check:", computed)
tree(store.root, "_INDEX/")

## 3 · Partly stale: adopt some, compute the rest

The source gains two rows on 2026-02-03 after the preload. The live row-check sees the new count,
the February window no longer fits, and only February is recomputed.

In [ ]:
left_csv = data / "left" / "ORDERS.csv"
frame = pd.read_csv(left_csv, dtype=str)
extra = frame[frame.BUSINESS_DATE == "2026-02-03"].head(2).assign(ORDER_ID=["999101", "999102"])
pd.concat([frame, extra]).to_csv(left_csv, index=False)
(data / "right" / "orders.csv").write_text(
    (data / "right" / "orders.csv").read_text()
    + "".join(
        f"2026-02-03,{row.ORDER_ID},{row.AMOUNT},{row.STATUS},{row.REGION}\n"
        for row in extra.itertuples()
    )
)
store, decisions, computed, partial_verdicts = run("partly_stale", known)
table(decisions)
print("computed by col-check:", computed)

## 4 · Same answer as a run without any preload

In [ ]:
_, _, plain_computed, plain_verdicts = run("plain")
print("plain run computed:", plain_computed)
print("partly-stale run == plain run:", partial_verdicts == plain_verdicts)
table(
    pd.DataFrame(
        [
            {"window": w, "column": c, "verdict": v[0]}
            for (w, c), v in sorted(partial_verdicts.items())
        ]
    )
)

## Rules

- **Adopt after Stage 1, before `col-check`.** `adopt_preloaded` refuses to run until
  `row_check/_CLOSED` exists, because it needs the manifests and the live counts.
- **Stage 1 always runs live** when the source is reachable; its counts are the evidence that a
  preload is still valid.
- **Never write `_INDEX/` by hand.** Adoption uses `commit_unit` (payload first, pointer last) with
  sequences after the highest existing one.
- **Never change a unit inside a run** once the Right side may have compared it. To fix a run,
  re-preload and start a new run.
- **One writer per pair** in the preload area; S3 has no locks.
- **Fixed-N pairs** are not preloadable; they go through the normal flow.
- **Only `col_map` columns are preloaded** (the keys of `col_map`, or the `columns` list of a
  Left-only config, minus the date column). If the run's `col_map` has more columns, the extra
  ones show `compute: no preload`; if it has fewer, the unused preloaded units are simply ignored.